In [ ]:
import json

from qiskit.circuit.library import U1Gate, U2Gate, U3Gate, SXGate


def load_ecc_library(json_path):
    with open(json_path) as f:
        raw_data = json.load(f)
    ecc_entries = raw_data[1]

    ecc_list_class_gates = dict()

    for class_key in ecc_entries:
        for entry in ecc_entries[class_key]:
            ecc_list_class_gates.setdefault(class_key, []).append(entry[1])

    ecc_map_transform = dict()

    for class_key in ecc_list_class_gates:
        patterns = ecc_list_class_gates[class_key]

        for i, pattern in enumerate(patterns):
            pattern_key = tuple([tuple([tuple(op) if isinstance(op, list) else op for op in gate]) for gate in pattern])

            others = [p for j, p in enumerate(patterns) if j != i]

            ecc_map_transform[pattern_key] = others

    return ecc_map_transform


In [ ]:
import qiskit
from qiskit.dagcircuit import DAGCircuit
from qiskit.circuit.library.standard_gates import XGate, HGate, CXGate, RZGate, SXGate, IGate, U1Gate, U2Gate, U3Gate


def nx_to_qiskit_dag(G_nx, num_qubits):
    dag = DAGCircuit()
    dag.add_qreg(qiskit.circuit.QuantumRegister(num_qubits, name='q'))

    id_to_op = {
        'i': IGate,
        'x': XGate,
        'h': HGate,
        'cx': CXGate,
        'rz': RZGate,
        'sx': SXGate,
        'u1': U1Gate,
        'u2': U2Gate,
        'u3': U3Gate,
    }

    for node_id in sorted(G_nx.nodes()):
        label = G_nx.nodes[node_id]['label']
        gate_cls = id_to_op.get(label.lower())
        if gate_cls is None:
            raise ValueError(f"Unknown gate: {label}")
        if 'qargs' in G_nx.nodes[node_id]:
            qubits = [dag.qubits[q] for q in G_nx.nodes[node_id]['qargs']]
        else:
            qubits = [dag.qubits[0]]
        dag.apply_operation_back(gate_cls(), qubits)

    return dag


In [ ]:
from qiskit.converters import dag_to_circuit
from qiskit.qasm2 import dumps


def save_optimized_circuit(G, num_qubits, original_qasm_file):
    dag = nx_to_qiskit_dag(G, num_qubits)
    qc_optimized = dag_to_circuit(dag)
    output_name = f"./output/nam/optimized_{os.path.basename(original_qasm_file)}"
    with open(output_name, "w") as f:
        f.write(dumps(qc_optimized))
    print(f"Optimized circuit saved: {output_name}")

In [ ]:
def find_matches(G, ecc_library):
    matches = []
    for src_pattern, dst_patterns in ecc_library.items():
        P = nx.DiGraph()
        for idx, gate in enumerate(src_pattern):
            P.add_node(idx, label=gate[0])
            if idx > 0:
                P.add_edge(idx - 1, idx)

        matcher = nx.isomorphism.DiGraphMatcher(
            G, P,
            node_match=lambda n1, n2: n1['label'] == n2['label']
        )

        for subgraph in matcher.subgraph_isomorphisms_iter():
            matches.append((src_pattern, dst_patterns, subgraph))
    return matches


In [ ]:
import os
import csv
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_dag
from torch_geometric.data import Data
from torch_geometric.nn import MessagePassing


def quantum_circuit_to_nx_dag(qc):
    dag = circuit_to_dag(qc)
    G = nx.DiGraph()
    node_id_map = {}
    for idx, node in enumerate(dag.topological_op_nodes()):
        G.add_node(idx, label=node.name)
        node_id_map[node._node_id] = idx
    for node in dag.topological_op_nodes():
        for succ in dag.quantum_successors(node):
            if isinstance(succ, type(node)):
                src = node_id_map[node._node_id]
                dst = node_id_map[succ._node_id]
                G.add_edge(src, dst)
    return G


def nx_graph_to_pyg_data(G):
    labels = nx.get_node_attributes(G, 'label')
    label_set = sorted(set(labels.values()))
    label_to_id = {label: idx for idx, label in enumerate(label_set)}
    x = torch.tensor([[label_to_id[labels[n]]] for n in G.nodes()], dtype=torch.float)
    edge_index = torch.tensor(list(G.edges()), dtype=torch.long).t().contiguous()
    if edge_index.numel() == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
    return Data(x=x, edge_index=edge_index)


class StableMPNN(MessagePassing):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(StableMPNN, self).__init__(aggr='add')
        self.msg_mlp = nn.Sequential(
            nn.Linear(in_channels * 2, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU()
        )
        self.update_mlp = nn.Sequential(
            nn.Linear(in_channels + hidden_channels, out_channels),
            nn.ReLU()
        )
        self.norm = nn.LayerNorm(out_channels)

    def forward(self, x, edge_index):
        return self.propagate(edge_index, x=x)

    def message(self, x_i, x_j):
        msg_input = torch.cat([x_i, x_j], dim=1)
        msg = self.msg_mlp(msg_input)
        return msg

    def update(self, aggr_out, x):
        out = self.update_mlp(torch.cat([x, aggr_out], dim=1))
        return self.norm(out)


class AgentPolicy(nn.Module):
    def __init__(self, embedding_dim, num_actions):
        super(AgentPolicy, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_actions)
        )

    def forward(self, embeddings):
        logits = self.net(embeddings)
        return F.softmax(logits, dim=-1)


class CentralizedCritic(nn.Module):
    def __init__(self, embedding_dim):
        super(CentralizedCritic, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, emb):
        return self.net(emb)


def validate_and_fix_probs(probs, agent_name="AGENT"):
    if torch.isnan(probs).any():
        print(f"NaN in probs! {agent_name}")
    row_sums = probs.sum(dim=-1, keepdim=True)
    probs = probs / row_sums.clamp(min=1e-8)
    probs = torch.where(torch.isnan(probs), torch.zeros_like(probs), probs)
    return probs


def compute_gae(rewards, values, gamma=0.98, lam=0.99):
    advantages = []
    gae = 0
    next_value = 0
    for t in reversed(range(len(rewards))):
        delta = rewards[t] + gamma * next_value - values[t]
        gae = delta + gamma * lam * gae
        advantages.insert(0, gae)
        next_value = values[t]
    returns = [a + v for a, v in zip(advantages, values)]
    return torch.tensor(returns, dtype=torch.float), torch.tensor(advantages, dtype=torch.float)


def ppo_step_policy_only(policy, optimizer_p,
                         embeddings, actions, old_log_probs, advantages,
                         clip_eps=0.1, entropy_coeff=0.005):
    probs = policy(embeddings)
    dist = torch.distributions.Categorical(probs)
    log_probs = dist.log_prob(actions)
    entropy = dist.entropy().mean()

    ratio = torch.exp(log_probs - old_log_probs)
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * advantages
    policy_loss = -torch.min(surr1, surr2).mean() - entropy_coeff * entropy

    optimizer_p.zero_grad()
    policy_loss.backward(create_graph=True)
    optimizer_p.step()

    return dict(
        policy_loss=policy_loss.item(),
        entropy=entropy.item(),
        ratio_mean=ratio.mean().item()
    )


def transform_selector(matches, actions):
    applied = []
    used_nodes = set()
    for match, action in zip(matches, actions):
        if action == 0:
            continue
        _, _, subgraph = match
        nodes = set(subgraph.keys())
        if nodes & used_nodes:
            continue
        applied.append((match, action))
        used_nodes |= nodes
    return applied


def apply_transformations(G, applied):
    G_new = G.copy()
    for match, action in applied:
        src_pattern, dst_patterns, subgraph = match
        dst_pattern = dst_patterns[action - 1]
        G_new.remove_nodes_from(list(subgraph.keys()))
        offset = max(G_new.nodes, default=-1) + 1
        for idx, gate in enumerate(dst_pattern):
            label = gate[0]
            G_new.add_node(offset + idx, label=label)
            if idx > 0:
                G_new.add_edge(offset + idx - 1, offset + idx)
    return G_new


def replace_subgraph_in_global(G_global, old_nodes, G_local_new):
    incoming, outgoing = [], []
    for node in old_nodes:
        for pred in G_global.predecessors(node):
            if pred not in old_nodes:
                incoming.append(pred)
        for succ in G_global.successors(node):
            if succ not in old_nodes:
                outgoing.append(succ)

    G_global.remove_nodes_from(old_nodes)

    offset = max(G_global.nodes, default=-1) + 1
    node_map = {}
    for old_id in G_local_new.nodes():
        new_id = offset + old_id
        node_map[old_id] = new_id
        G_global.add_node(new_id, **G_local_new.nodes[old_id])

    for src, dst in G_local_new.edges():
        G_global.add_edge(node_map[src], node_map[dst])

    new_nodes = list(node_map.values())
    for pred in incoming:
        G_global.add_edge(pred, new_nodes[0])
    for succ in outgoing:
        G_global.add_edge(new_nodes[-1], succ)

    return G_global


def safe_mean(tensor, dim=0, keepdim=True, fallback_dim=None):
    if tensor.numel() == 0:
        if fallback_dim is None:
            raise ValueError("safe_mean fallback_dim is required for empty tensor.")
        return torch.zeros((1, fallback_dim), dtype=tensor.dtype, device=tensor.device)
    mean = tensor.mean(dim=dim, keepdim=keepdim)
    if torch.isnan(mean).any():
        mean = torch.zeros_like(mean)
    return mean

def train_step_multiagent_local(qc, mpnn, agents, critic, optimizers_p, optimizer_c, ecc_library,
                                alpha=0.5, beta=0.35, tau=0.15):
    G = quantum_circuit_to_nx_dag(qc)
    initial_size = len(G.nodes)
    initial_depth = nx.dag_longest_path_length(G)
    initial_cx = sum(1 for _, data in G.nodes(data=True) if data.get("label") == "cx")

    data = nx_graph_to_pyg_data(G)
    node_embeddings = mpnn(data.x, data.edge_index)

    partitions = list(nx.algorithms.community.greedy_modularity_communities(G))
    local_applied = []

    for i, (key, agent) in enumerate(agents.items()):
        if i >= len(partitions):
            continue

        local_nodes = list(partitions[i])
        G_local = G.subgraph(local_nodes).copy()
        matches = find_matches(G_local, ecc_library)
        if not matches:
            continue

        emb_list = []
        for m in matches:
            nodes = list(m[2].keys())
            embedding_dim = node_embeddings.shape[1]
            match_emb = safe_mean(
                node_embeddings[nodes],
                dim=0, keepdim=True,
                fallback_dim=embedding_dim
            )
            emb_list.append(match_emb)
        emb = torch.cat(emb_list, dim=0)
        probs = validate_and_fix_probs(agent(emb), agent_name=key)
        dist = torch.distributions.Categorical(probs)
        actions = dist.sample()
        old_log_probs = dist.log_prob(actions)
        applied = transform_selector(matches, actions.tolist())
        if applied:
            local_applied.append((G_local, local_nodes, applied, emb, actions, old_log_probs))

    for G_local, local_nodes, applied, *_ in local_applied:
        G_new = apply_transformations(G_local, applied)
        G = replace_subgraph_in_global(G, local_nodes, G_new)

    G = nx.convert_node_labels_to_integers(G)

    final_size = len(G.nodes)
    final_depth = nx.dag_longest_path_length(G)
    final_cx = sum(1 for _, data in G.nodes(data=True) if data.get("label") == "cx")

    r = alpha * (initial_size - final_size) + beta * (initial_depth - final_depth) + tau * (initial_cx - final_cx)
    print(
        f"Global Reward: {r:.4f} | size: {initial_size}->{final_size}, depth: {initial_depth}->{final_depth}, cx: {initial_cx}->{final_cx}")

    full_data = nx_graph_to_pyg_data(G)
    full_emb = mpnn(full_data.x, full_data.edge_index)
    V = critic(full_emb.mean(dim=0, keepdim=True))
    returns, advantages = compute_gae([r], V)

    for i, (_, _, _, emb, actions, old_log_probs) in enumerate(local_applied):
        key = list(agents.keys())[i]
        result = ppo_step_policy_only(agents[key], optimizers_p[key], emb, actions, old_log_probs,
                                      advantages.expand_as(actions))

    optimizer_c.zero_grad()
    V_loss = F.mse_loss(V, torch.tensor([r], dtype=V.dtype))
    V_loss.backward()
    optimizer_c.step()

    print(f"Final Critic Loss: {V_loss.item():.4f}")

    metrics = dict(
        reward=r,
        size_old=initial_size, size_new=final_size,
        depth_old=initial_depth, depth_new=final_depth,
        cx_old=initial_cx, cx_new=final_cx
    )
    return local_applied, metrics, G


def train_loop_multiagent_local(dataset, mpnn, agents, critic, optimizers_p, optimizer_c, ecc_library, num_epochs=5):
    log_file = "./training_log_local_multiagent.csv"
    with open(log_file, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['epoch', 'qasm_file', 'total_reward', 'size_old', 'size_new', 'depth_old', 'depth_new'])

    for epoch in range(1, num_epochs + 1):
        total_reward = 0.0
        for qasm_file in dataset:
            qc = QuantumCircuit.from_qasm_file(qasm_file)
            agent_results, metrics, final_G = train_step_multiagent_local(
                qc, mpnn, agents, critic, optimizers_p, optimizer_c, ecc_library)
            if agent_results and metrics:
                total_reward += metrics['reward']
                with open(log_file, mode='a', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerow([
                        epoch, os.path.basename(qasm_file),
                        metrics['reward'],
                        metrics['size_old'],
                        metrics['size_new'],
                        metrics['depth_old'],
                        metrics['depth_new']
                    ])
            final_G = nx.convert_node_labels_to_integers(final_G)
            save_optimized_circuit(final_G, qc.num_qubits, qasm_file)

        print(f"[Epoch {epoch}] Total reward: {total_reward:.4f}")
        for key, agent in agents.items():
            torch.save(agent.state_dict(), f"./pt/agent/agent_{key}_epoch_{epoch}.pt")
        torch.save(critic.state_dict(), f"./pt/critic/critic_epoch_{epoch}.pt")

    print(f"Training done. Log saved to {log_file}")


In [ ]:
agents = {}
for i in range(16):
    agents[f'agent_{i}'] = AgentPolicy(embedding_dim=4, num_actions=2)

optimizers_p = {key: torch.optim.Adam(agent.parameters(), lr=3e-4) for key, agent in agents.items()}
critic = CentralizedCritic(embedding_dim=4)
optimizer_c = torch.optim.Adam(critic.parameters(), lr=1e-3)
mpnn = StableMPNN(in_channels=1, hidden_channels=4, out_channels=4)

dataset_nam = [
    "../circuits/nam/test.qasm",
    "../circuits/nam/adder_8.qasm",
    "../circuits/nam/barenco_tof_3.qasm",
    "../circuits/nam/barenco_tof_4.qasm",
    "../circuits/nam/barenco_tof_5.qasm",
    "../circuits/nam/barenco_tof_10.qasm",
    "../circuits/nam/csla_mux_3.qasm",
    "../circuits/nam/csum_mux_9.qasm",
    "../circuits/nam/gf2^4_mult.qasm",
    "../circuits/nam/gf2^5_mult.qasm",
    "../circuits/nam/gf2^6_mult.qasm",
    "../circuits/nam/gf2^7_mult.qasm",
    "../circuits/nam/gf2^8_mult.qasm",
    "../circuits/nam/gf2^9_mult.qasm",
    "../circuits/nam/gf2^10_mult.qasm",
    "../circuits/nam/gf2^16_mult.qasm",
    "../circuits/nam/gf2^32_mult.qasm",
    "../circuits/nam/grover_5.qasm",
    "../circuits/nam/ham15-high.qasm",
    "../circuits/nam/ham15-low.qasm",
    "../circuits/nam/ham15-med.qasm",
    "../circuits/nam/hwb6.qasm",
    "../circuits/nam/mod5_4.qasm",
    "../circuits/nam/mod_adder_1024.qasm",
    "../circuits/nam/mod_mult_55.qasm",
    # "../circuits/nam/mod_red_21.qasm",
    # "../circuits/nam/qcla_adder_10.qasm",
    # "../circuits/nam/qcla_com_7.qasm",
    # "../circuits/nam/qcla_mod_7.qasm",
    # "../circuits/nam/rc_adder_6.qasm",
    # "../circuits/nam/tof_3.qasm",
    # "../circuits/nam/tof_4.qasm",
    # "../circuits/nam/tof_5.qasm",
    # "../circuits/nam/tof_10.qasm",
    # "../circuits/nam/vbe_adder_3.qasm",
]

dataset_ibm = [
    "../circuits/ibm/test.qasm",
    "../circuits/ibm/adder_8.qasm",
    "../circuits/ibm/barenco_tof_3.qasm",
    "../circuits/ibm/barenco_tof_4.qasm",
    "../circuits/ibm/barenco_tof_5.qasm",
    "../circuits/ibm/barenco_tof_10.qasm",
    "../circuits/ibm/csla_mux_3.qasm",
    "../circuits/ibm/csum_mux_9.qasm",
    "../circuits/ibm/gf2^4_mult.qasm",
    "../circuits/ibm/gf2^5_mult.qasm",
    "../circuits/ibm/gf2^6_mult.qasm",
    "../circuits/ibm/gf2^7_mult.qasm",
    "../circuits/ibm/gf2^8_mult.qasm",
    "../circuits/ibm/gf2^9_mult.qasm",
    "../circuits/ibm/gf2^10_mult.qasm",
    "../circuits/ibm/gf2^16_mult.qasm",
    "../circuits/ibm/gf2^32_mult.qasm",
    "../circuits/ibm/grover_5.qasm",
    "../circuits/ibm/ham15-high.qasm",
    "../circuits/ibm/ham15-low.qasm",
    "../circuits/ibm/ham15-med.qasm",
    "../circuits/ibm/hwb6.qasm",
    "../circuits/ibm/mod5_4.qasm",
    "../circuits/ibm/mod_adder_1024.qasm",
    "../circuits/ibm/mod_mult_55.qasm",
    "../circuits/ibm/mod_red_21.qasm",
    "../circuits/ibm/qcla_adder_10.qasm",
    "../circuits/ibm/qcla_com_7.qasm",
    "../circuits/ibm/qcla_mod_7.qasm",
    "../circuits/ibm/rc_adder_6.qasm",
    "../circuits/ibm/tof_3.qasm",
    "../circuits/ibm/tof_4.qasm",
    "../circuits/ibm/tof_5.qasm",
    "../circuits/ibm/tof_10.qasm",
    "../circuits/ibm/vbe_adder_3.qasm",
    # "../circuits/ibm/vqe_nativegates_ibm_tket_8",
    # "../circuits/ibm/qgan_nativegates_ibm_tket_8",
    # "../circuits/ibm/qaoa_nativegates_ibm_tket_8",
    # "../circuits/ibm/ae_nativegates_ibm_tket_8",
    # "../circuits/ibm/qpeexact_nativegates_ibm_tket_8",
    # "../circuits/ibm/qpeinexact_nativegates_ibm_tket_8",
    # "../circuits/ibm/qft_nativegates_ibm_tket_8",
    # "../circuits/ibm/qftentangled_nativegates_ibm_tket_8",
    # "../circuits/ibm/portfoliovqe_nativegates_ibm_tket_8",
    # "../circuits/ibm/portfolioqaoa_nativegates_ibm_tket_8",
]


ecc_library_nam = load_ecc_library("../ecc_sets/nam_325_ecc.json")
ecc_library_ibm = load_ecc_library("../ecc_sets/ibm_325_ecc.json")

train_loop_multiagent_local(
    dataset_nam, mpnn, agents, critic,
    optimizers_p, optimizer_c, ecc_library_nam,
    num_epochs=10
)